[Reference](https://python.plainenglish.io/i-built-an-open-source-library-fastapi-middlewares-f85225a15955)

# Installation

In [2]:
!pip install fastapi-middlewares
# uv add fastapi-middlewares

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.7/110.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 kB 19.2 MB/s eta 0:00:00
  Attempting uninstall: starlette
    Found existing installation: starlette 0.48.0
    Uninstalling starlette-0.48.0:
      Successfully uninstalled starlette-0.48.0
  Attempting uninstall: fastapi
    Found existing installation: fastapi 0.118.3
    Uninstalling fastapi-0.118.3:
      Successfully uninstalled fastapi-0.118.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.19.0 requires fastapi<0.119.0,>=0.115.0, but you have fastapi 0.122.0 which is incompatible.


# Quick Start

In [3]:
from fastapi import FastAPI
from middlewares import add_essentials

app = FastAPI()

# Add all essential middlewares in one line
add_essentials(app)

@app.get("/")
def root():
    return {"message": "Hello World"}

# 1. Request ID Middleware

In [4]:
from fastapi import FastAPI, Request
from middlewares import RequestIDMiddleware

app = FastAPI()
app.add_middleware(RequestIDMiddleware)


@app.get("/users/{user_id}")
def get_user(user_id: int, request: Request):
    request_id = request.scope.get("request_id")
    return {"user_id": user_id, "request_id": request_id}

# 2. Request Timing Middleware

In [5]:
from middlewares import RequestTimingMiddleware

app.add_middleware(RequestTimingMiddleware)

# 3. Security Headers Middleware


In [6]:
from middlewares import SecurityHeadersMiddleware

app.add_middleware(SecurityHeadersMiddleware)

# 4. Logging Middleware

In [7]:
from middlewares import LoggingMiddleware

app.add_middleware(
    LoggingMiddleware,
    logger_name="my_app",
    skip_paths=["/health", "/metrics"]
)

# 5. Error Handling Middleware

In [8]:
from middlewares import ErrorHandlingMiddleware


app.add_middleware(
    ErrorHandlingMiddleware,
    include_traceback=False  # Set True for development
)

In [9]:
from starlette.responses import JSONResponse


async def handle_value_error(scope, exc):
    return JSONResponse(
        status_code=400,
        content={"error": "bad_request", "message": str(exc)}
    )


app.add_middleware(
    ErrorHandlingMiddleware,
    custom_handlers={ValueError: handle_value_error}
)

# 6. CORS Middleware

In [11]:
from middlewares import add_cors

add_cors(
    app,
    allow_origins=["http://localhost:3000"],
    # allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)

# 7. GZip Compression

In [12]:
from middlewares import add_gzip

add_gzip(app, minimum_size=1000)

# Middleware Ordering

In [13]:
from fastapi import FastAPI
from middlewares import (
    ErrorHandlingMiddleware,
    SecurityHeadersMiddleware,
    RequestIDMiddleware,
    RequestTimingMiddleware,
    LoggingMiddleware,
    add_cors,
    add_gzip,
)


app = FastAPI()


# Last added = First executed
add_gzip(app)                               # 7. Compress response
app.add_middleware(LoggingMiddleware)       # 6. Log request/response
app.add_middleware(RequestTimingMiddleware) # 5. Time request
app.add_middleware(RequestIDMiddleware)     # 4. Add request ID
app.add_middleware(SecurityHeadersMiddleware) # 3. Add security headers
add_cors(app)                               # 2. Handle CORS
app.add_middleware(ErrorHandlingMiddleware) # 1. Catch errors (outermost)

# Complete Example

In [14]:
from fastapi import FastAPI, HTTPException
from middlewares import add_essentials

app = FastAPI(title="My API")


# Add all middlewares with custom config
add_essentials(
    app,
    cors_origins=["http://localhost:3000"],
    include_traceback=False,  # Set True for development
    logger_name="my_api"
)

@app.get("/")
def root():
    return {"message": "Hello World"}



@app.get("/users/{user_id}")
def get_user(user_id: int):
    if user_id < 1:
        raise ValueError("Invalid user ID")
    return {"user_id": user_id, "name": "John"}


@app.get("/error")
def error():
    raise HTTPException(status_code=404, detail="Not found")

```
uvicorn main:app --reload

# Check headers
curl -I http://localhost:8000/
```